1.  Removed MaxPooling (was destroying time info)
2. Used padding='same'
3. Lightweight CNN (not over-compressing)
4. Proper architecture for time-series

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, LSTM, Dense, Dropout,
    BatchNormalization, Input, Activation
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import os

In [ ]:
path = "/content/drive/MyDrive/Sensor_DL/final_data"

X_train = np.load(os.path.join(path, "X_train.npy"))
y_train = np.load(os.path.join(path, "y_train.npy"))

X_val = np.load(os.path.join(path, "X_val.npy"))
y_val = np.load(os.path.join(path, "y_val.npy"))

X_test = np.load(os.path.join(path, "X_test.npy"))
y_test = np.load(os.path.join(path, "y_test.npy"))

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (24665, 50, 21)
Val: (2700, 50, 21)
Test: (6818, 50, 21)


In [ ]:
y_train -= 1
y_val -= 1
y_test -= 1

num_classes = len(np.unique(y_train))


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))


In [ ]:
model = Sequential([
    Input(shape=(50, X_train.shape[2])),

    # CNN Block (64 → 128)
    Conv1D(64, kernel_size=3, padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),

    Conv1D(128, kernel_size=3, padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),

    Conv1D(128, kernel_size=1, activation='relu'),

    # LSTM
    LSTM(64),

    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_7 (Conv1D)               │ (None, 50, 64)         │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 50, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 50, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 50, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 50, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,428 (392.30 KB)

 Trainable params: 100,044 (390.80 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/Sensor_DL/best_cnn_lstm_fixed_3.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

Epoch 1/30
94/97 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1680 - loss: 2.3594
Epoch 1: val_accuracy improved from None to 0.36333, saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_lstm_fixed_3.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_lstm_fixed_3.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.2570 - loss: 2.1239 - val_accuracy: 0.3633 - val_loss: 1.8713
Epoch 2/30
95/97 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4967 - loss: 1.4068
Epoch 2: val_accuracy improved from 0.36333 to 0.66963, saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_lstm_fixed_3.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_lstm_fixed_3.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5478 - loss: 1.2791 - val_accuracy: 0.6696 - val_loss: 1.0062
Epoch 3/30
95/97 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6668 - loss: 0.9656
Epoch 3: val_accuracy improved from 0.66963 to 0.74296, sa

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("\n Fixed CNN-LSTM Test Accuracy:", acc)

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9368 - loss: 0.1797

 Fixed CNN-LSTM Test Accuracy: 0.9367849826812744


In [ ]:
model.save("/content/drive/MyDrive/Sensor_DL/final_cnn_lstm_fixed_3.keras")

print(" Model saved successfully!")

 Model saved successfully!
